In [ ]:
!python --version

In [ ]:
import os, sys
import pandas as pd

from pathlib import Path
from sklearn.metrics import pairwise_distances


ROOT = Path().resolve().parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.calc_degs_lib import CALC_DEGS
from libs.tcga_gdc_lib import *
from libs.Basic import *
from libs.stat_lib import *


### Defaults

In [ ]:
ROOT = Path().resolve().parent
root0 = ROOT / "data"

gdc = GDC(root0=root0)

os.listdir(root0)[:10]


### Get all programs

In [ ]:
force=False
verbose=True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)


In [ ]:
np.array(prog_list)

### Primary sites given a program

In [ ]:
gdc.url_gdc_project

In [ ]:
prog_id = 'TCGA'

gdc.set_program(prog_id)

In [ ]:
df_psi = gdc.get_primary_sites(prog_id=prog_id, force=force, verbose=verbose)

primary_sites = np.unique(df_psi.primary_site.to_list())
primary_sites[:6]

In [ ]:
force=False
verbose=False

primary_site = 'Breast'

df_cases, df_all_samples, df_all_mut, barcode_list = gdc.get_filtered_tables(primary_site=primary_site, verbose=verbose)

print(len(df_all_mut))
df_all_mut.columns


In [ ]:
cols = ['case_id', 'psi_id', 'primary_site', 'disease_type',  'diagnoses', 
       'subtype_global', 'stage_ajcc', 'primary_diagnosis', 'tumor_grade',
        'tumor_stage', 'stage', 'tumor_class', 'histology',
       'subtype_tissue'] # 'stage_clin', 'figo_stage',

print(len(df_cases))
df_cases[cols].tail(3)

In [ ]:
min_barcodes=3
min_genes=3

dfpiv = gdc.build_pivot_table(df_all_mut, min_barcodes=min_barcodes, min_genes=min_genes)
print(len(dfpiv))
dfpiv.head(2)

### Purity

In [ ]:
cluster_analysis = 'HDBSCAN'
cluster_analysis = 'UMAP'

k = None
min_cluster_size=None
min_samples=None

if cluster_analysis == 'UMAP':
    k=5
    print(f"Chose {cluster_analysis} with k={k}")
    _, labels = gdc.calc_UMAP(dfpiv, k)

elif cluster_analysis == 'HDBSCAN':
    min_cluster_size=5
    min_samples=3
    print(f"Chose {cluster_analysis} with min_cluster_size={min_cluster_size} and min_samples={min_samples}")

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        _, labels, _ = gdc.calc_HDBSCAN(dfpiv, min_cluster_size=min_cluster_size, min_samples=min_samples)
else:
    print("Did not defined the clustering method")

In [ ]:
from sklearn.metrics import pairwise_distances

lab_list = np.unique(labels)
pu_list = []
n_list = []
pairs = []

for label in lab_list:
    idx = labels == label
    X = dfpiv[idx].to_numpy(dtype=bool)

    n_bardodes = len(X)
    n_list.append(n_bardodes)
    pairs.append(n_bardodes * (n_bardodes - 1) / 2)

    if len(X) > 1:
        dist = pairwise_distances(X, metric="jaccard")
        similarity = 1 - dist
        purity = similarity[np.triu_indices_from(similarity, k=1)].mean()
    else:
        purity = 0

    # print(label, purity)
    pu_list.append(np.round(purity, 3))


dfa = pd.DataFrame({
    "label": lab_list,
    "n_barcodes": n_list,
    "purity": pu_list,
    "n_pairs": pairs
})

max_pairs = dfa["n_pairs"].max()
dfa["purity_norm"] = dfa["purity"] * (dfa["n_pairs"] / max_pairs)

dfa = dfa.sort_values(by="purity_norm", ascending=False)

dfa

In [ ]:
# df = dfpiv.copy()
dfpiv["cluster"] = labels

dfclu = dfpiv.groupby("cluster").mean().T

gene_score = dfclu.sum(axis=1)  # sum across clusters
dfclu = dfclu.loc[gene_score.sort_values(ascending=False).index]

dfclu.head(8)

In [ ]:
threshold = 0.05

good_clusters = dfa.loc[dfa["purity_norm"] >= threshold, "label"]

# dfclu_filtered = dfclu[good_clusters]
# dfclu_filtered.head(3)

good_clusters = [x for x in good_clusters if x != -1]
good_clusters

### Barplots per label

In [ ]:
min_perc = 0.10
# top_n_genes = 30

ngood = len(good_clusters)
nrows = int(np.ceil(ngood / 2))

height = 6*nrows
    
fig, axes = plt.subplots(nrows, 2, figsize=(12, height))
axes = axes.flatten()

for ax, label in zip(axes, good_clusters):
    if label == -1:
        continue
    
    dfb = dfclu[label]

    dfb = dfb[ dfb.values > min_perc]
    dfb = dfb.sort_values(ascending=False)
    # dfb = dfb.sort_values(ascending=False).head(top_n_genes)
    
    ax.bar(dfb.index, dfb.values)

    stri = f"Label {label} | impurity={dfa.loc[dfa['label']==label].iloc[0].purity_norm:.3f}"

    ax.set_title(stri)
    ax.set_ylabel("Representative percentage")
    ax.set_xlabel("Genes")
    ax.tick_params(axis="x", rotation=70)

    print(stri, ", ".join(dfb.index.to_list()))

plt.tight_layout()
plt.show()


### Enrichment Analysis

In [ ]:
SUBTYPE_GENES = {
    "Luminal_A": {
        "PIK3CA","GATA3","MAP3K1","CDH1","TBX3","RUNX1","FOXA1"
    },
    "Luminal_B": {
        "PIK3CA","TP53","GATA3","RB1","CCND1","ERBB2"
    },
    "HER2": {
        "ERBB2","PIK3CA","TP53","PTEN"
    },
    "TNBC_Basal": {
        "TP53","BRCA1","RB1","PTEN","NF1"
    },
    "Lobular": {
        "CDH1","PIK3CA","FOXA1","TBX3","GATA3","MAP3K1"
    }
}

In [ ]:
from scipy.stats import hypergeom

def enrichment_test(sample_genes, subtype_genes, background_genes):
    N = len(background_genes)
    K = len(subtype_genes)
    n = len(sample_genes)
    overlap = len(set(sample_genes) & set(subtype_genes))

    # P(X >= k)
    pval = hypergeom.sf(overlap - 1, N, K, n)

    return overlap, pval

In [ ]:
background_genes = np.unique(df_all_mut.symbol.to_list())

print(len(background_genes))
background_genes[:5]

In [ ]:
lista = []
for label in good_clusters:
    
    impurity = dfa.loc[dfa['label']==label].iloc[0].purity_norm
              
    dfb = dfclu[label]

    dfb = dfb[ dfb.values > min_perc]
    # dfb = dfb.sort_values(ascending=False)

    sample_genes = dfb.index.to_list()

    # stri = f"Label {label} | impurity={impurity:.3f}"
    # print(stri, ", ".join(sample_genes))

    for subtype, annotated_genes in SUBTYPE_GENES.items():
        overlap, pval = enrichment_test(sample_genes, annotated_genes, background_genes)
        # print(f"Subtype: {subtype}, overlap: {overlap}, p-value: {pval}")

        if overlap >= 2:
            mat = [label, impurity, subtype, overlap, pval]
            lista.append(mat)

df = pd.DataFrame(lista, columns=["label", "impurity", "subtype", "overlap", "pval"])
df['fdr'] = fdr(df['pval'])
df

### Heatmap is wrong

In [ ]:
import seaborn as sns
plt.figure(figsize=(10, 12))

N = 30

sns.heatmap(
    dfclu.head(N),
    cmap="viridis",
    linewidths=0.5,
    linecolor="lightgray"
)

plt.title(f"Top {N} representative genes per cluster")
plt.xlabel("Clusters")
plt.ylabel("Genes")

plt.tight_layout()
plt.show()